# Prepare ERA5 Data for Wind Workflow

This example shows how to use RESKit's reproducible ERA5 preparation workflow.
It downloads ERA5 data from the Copernicus Climate Data Store (CDS), applies
preprocessing (wind speed from u/v components, solar unit conversions), and
optionally tiles the result into the `<zoom>/<x>/<y>/<year>/` directory structure
expected by `Era5Source` and `execute_workflow_iteratively()`.

**Prerequisites**
- CDS account and `~/.cdsapirc` API key configured (for instructions see link below):
  https://cds.climate.copernicus.eu/how-to-api
  (processing is done with xarray/netCDF4 — no external CDO binary required)

In [1]:
from pathlib import Path

import reskit as rk

## Step 1 – Download, process, and tile in one call

`download_and_process` orchestrates all three steps:
1. Downloads the ERA5 variables required by the chosen workflow from CDS
2. Preprocesses them (ws100 from u/v components, solar unit + time-shift corrections)
3. Splits the processed files into the tiled directory structure

Output layout under `output_dir`:
```
output_dir/
├── raw/        ← CDS download files only
├── processed/  ← usable processed NC files (ws100, solar_t_adj, …)
└── tiles/      ← tiled output: <zoom>/<x-tile>/<y-tile>/<year>/*.nc
```

Set `tiling=False` to skip step 3; `era5_path` will then point to `processed/`
and can be used directly with Era5Source.

The wind workflow also depends on Global Wind Atlas (GWA) data. That source cannot
be downloaded automatically yet, so `download_and_process` prepares the ERA5 data
and just prints a note reminding you to fetch the GWA rasters manually (see Step 4).

In [2]:
# Path.cwd() resolves to the directory the notebook is run from;
# change it to a specific path if you want the output elsewhere.
output_dir = Path.cwd() / "tmp"

result = rk.download_and_process(
    workflow="wind_era5_PenaSanchezDunkelWinklerEtAl2025",
    start_date="2000-01-01",
    end_date="2000-01-03",
    # Boundary box around Germany. Note the extra western margin (west=4
    # rather than ~6): spatial interpolation needs a few grid cells on every
    # side of each placement, so the box must extend past the locations you
    # simulate in Step 4. ERA5 is 0.25 deg, and bilinear interpolation needs
    # 2 cells (cubic 4) of padding, i.e. at least ~0.5-1.0 deg beyond your
    # placements. Otherwise you get "Insufficient data" at simulation time.
    boundary_box={"north": 55, "south": 47, "west": 4, "east": 15},
    output_dir=output_dir,
    tiling=True,
    zoom_level=4,  # 16x16 global tile grid, standard for RESKit ERA5 data
)

# you can check the download status of the raw data at:
# https://cds.climate.copernicus.eu/requests

# NOTE: If you intend to use this outside of example, please make sure to always download a full year.

NOTE: Automated Global Wind Atlas (GWA4) download is not implemented yet. Download the required rasters manually from https://globalwindatlas.info/en/download/gis-files and pass them to the workflow (e.g. gwa_100m_path / height_scaling_data).


2026-07-09 13:10:48,725 INFO Request ID is 5cfe72e5-f231-4ea1-a5ed-b0a42375ba58
2026-07-09 13:10:48,796 INFO status has been updated to accepted
2026-07-09 13:11:09,952 INFO status has been updated to running
2026-07-09 13:11:21,387 INFO status has been updated to successful


710e61f89c6204039d8a0e32f34b59c8.nc:   0%|          | 0.00/1.02M [00:00<?, ?B/s]

ERA5 preprocessing done.
ERA5 tiling done.


## Step 2 – Inspect the result

When `tiling=True`, `era5_path` is a path template with `<ZOOM>`, `<X-TILE>`,
and `<Y-TILE>` placeholders. Pass it to `get_dataframe_with_weather_tilepaths()`
to resolve tile paths for a set of locations.

In [3]:
print("era5_path template:", result["era5_path"])
# → <script_dir>/tmp/tiles/<ZOOM>/<X-TILE>/<Y-TILE>

era5_path template: /fast/home/p-dunkel/playground/RESKit_HA/examples/1_load_input_data/tmp/tiles/<ZOOM>/<X-TILE>/<Y-TILE>


## Step 3 – Browse the downloaded files directly

For later simulation, you can use either the `processed/` or the tiled
`tiles/` output (if chosen in Step 1) — see the `output_dir`
layout described above. The `raw/` files are kept only for reference or
further processing and should not be fed into the simulation workflows.

With tiling enabled, the data is written as one `.nc` file per variable
per year, under `tiles/<zoom>/<x-tile>/<y-tile>/<year>/`. Below we pick one
of the tiles written above and open its wind speed file for the year we
downloaded, just to see what's in it.

In [4]:
import xarray as xr

year = 2000  # we only downloaded 2000 above

# pick any one tile directory for that year (there may be several, depending
# on how many tiles the boundary box spans)
tile_dir = sorted((output_dir / "tiles").glob(f"*/*/*/{year}"))[0]
print("Example tile directory:", tile_dir)

# each variable is stored in its own file, named
# "<...>.z<zoom>.x<x-tile>.y<y-tile>.y<year>.<parameter>.nc"
parameter_file = sorted(tile_dir.glob("*100m_wind_speed*.nc"))[0]
print("Example parameter file:", parameter_file)

with xr.open_dataset(parameter_file) as ds:
    print(ds)

Example tile directory: /fast/home/p-dunkel/playground/RESKit_HA/examples/1_load_input_data/tmp/tiles/4/8/5/2000
Example parameter file: /fast/home/p-dunkel/playground/RESKit_HA/examples/1_load_input_data/tmp/tiles/4/8/5/2000/reanalysis-era5-single-levels.z4.x8.y5.y2000.100m_wind_speed.processed.nc
<xarray.Dataset> Size: 856kB
Dimensions:    (time: 72, latitude: 33, longitude: 45)
Coordinates:
  * time       (time) datetime64[ns] 576B 2000-01-01 ... 2000-01-03T23:00:00
  * latitude   (latitude) float32 132B 55.0 54.75 54.5 54.25 ... 47.5 47.25 47.0
  * longitude  (longitude) float32 180B 4.0 4.25 4.5 4.75 ... 14.5 14.75 15.0
Data variables:
    ws100      (time, latitude, longitude) float64 855kB ...


## Step 4 – Use the data in a real wind simulation workflow

`execute_workflow_iteratively()` takes care of picking the right weather
tile for each placement, so the tiled `era5_path` template from Step 1 can
be handed straight to a RESKit wind workflow (here:
`wind_era5_PenaSanchezDunkelWinklerEtAl2025`). Note that the tile template
only resolves `<ZOOM>`/`<X-TILE>`/`<Y-TILE>` — the `<year>` subfolder
(see Step 3) still needs to be appended manually, since each simulation
run covers a single year of weather data. It also needs Global Wind Atlas
(GWA) rasters to bias-correct the ERA5 wind speeds; the small bundled
`rk.TEST_DATA` rasters are used below as stand-ins — swap them for real
GWA files (https://globalwindatlas.info/) for an actual study.

See `examples/3_wind/3_7_example_ethos_reskit_wind_workflow.ipynb` for
more on the workflow itself.

In [5]:
import pandas as pd

# Example: three locations in Germany, each with turbine specifications
placements = pd.DataFrame(
    {
        "lon": [6.004750, 6.1, 6.2],
        "lat": [50.784432, 50.8, 50.9],
        "hub_height": [120, 120, 82],
        "capacity": [4000, 4000, 4000],
        "rotor_diam": [130, 150, 136],
    }
)

reskit_xr = rk.execute_workflow_iteratively(
    workflow=rk.wind.wind_era5_PenaSanchezDunkelWinklerEtAl2025,
    weather_path_varname="era5_path",
    zoom=4,
    placements=placements,
    era5_path=str(Path(result["era5_path"]) / str(year)),
    gwa_100m_path=rk.TEST_DATA["gwa100-like.tif"],
    height_scaling_data={
        50: rk.TEST_DATA["gwa50-like.tif"],
        200: rk.TEST_DATA["gwa200-like.tif"],
    },
)
print(reskit_xr["capacity_factor"])

2026-07-09 13:11:23.106960 Now processing tile 1/1 with 3 locations: /fast/home/p-dunkel/playground/RESKit_HA/examples/1_load_input_data/tmp/tiles/4/8/5/2000


2026-07-09 13:11:25.789414 Now extracting correction factors for a total of 3 placements from /fast/home/p-dunkel/playground/RESKit_HA/reskit/wind/core/data/cf_correction_factors_PSDW2025.tif:
2026-07-09 13:11:25.792851 Based on max_batch_size=3, the total of 3 placements were split into 1 sub batches. Proceeding with batch 1/1 (id=0) with 3 placements.
2026-07-09 13:11:25.797091 Maximum rel. deviation after initial simulation is 0.2742, Number/share of placements with deviation > tolerance (0.01): 3/3. More iterations required.
2026-07-09 13:11:25.798987 Maximum rel. deviation after 1 additional iteration(s) is 0.0237, Number/share of placements with deviation > tolerance (0.01): 2/3. More iterations required.
2026-07-09 13:11:25.800496 Required tolerance of 0.01 reached after 2 additional iteration(s). Maximum remaining rel. deviation: 0.0031.
<xarray.DataArray 'capacity_factor' (time: 72, location: 3)> Size: 2kB
array([[0.07938, 0.08918, 0.06174],
       [0.07644, 0.08918, 0.06076],